In [1]:
# Parameters
summary_config = {"run_run_comparison": False, "run_RTP_summary": True, "run_validation": True, "run_network_validation": True, "summary_list": {"RTP-summary-notebook": ["RTP_index", "RTP_congestion", "RTP_topsheet", "RTP_MIC", "RTP_person", "RTP_household", "RTP_access", "RTP_costs", "RTP_walk_bike", "RTP_emissions", "RTP_mode_share", "RTP_freight", "RTP_transit"], "activitysim-validation-notebook": ["work_from_home", "auto_ownership", "telecommute_frequency", "free_parking", "cdap", "intermediate_stop_frequency", "trip_purpose", "trip_destination_choice", "school_location", "work_location", "mandatory_tour_frequency", "mandatory_tour_scheduling", "non_mandatory_tour_frequency", "non_mandatory_tour_destination_choice", "non_mandatory_tour_scheduling", "joint_tour_frequency", "joint_tour_composition", "atwork_subtours_frequency", "atwork_subtours_destination_choice", "atwork_subtours_scheduling", "atwork_subtour_mode", "tour_mode_choice", "trip_mode_choice"], "network-validation-notebook": ["JBLM", "supplementals", "transit_validation", "traffic_validation", "bike_validation", "link_analysis"], "run-comparison-notebook": ["topsheet", "population", "parking", "vmt", "transit"]}, "p_output_dir": "outputs/summary", "output_folder": "outputs", "survey_folder": "inputs/base_year/survey", "uncloned_folder": "uncloned", "sc_run_name": "current run", "sc_run_path": "../../../../", "survey_directories": {"survey": "../../../../inputs/base_year/survey"}, "comparison_runs_list": {"2050 new transit, old network": "\\\\modelstation3\\c$\\Workspace\\sc_new_2050_transit\\soundcast", "2050 urbansim": "\\\\modelstation2\\c$\\Workspace\\sc_2050_urbansim2_07_30_25"}, "county_map": {"33": "King", "35": "Kitsap", "53": "Pierce", "61": "Snohomish"}, "uc_list": ["@sov_inc1", "@sov_inc2", "@sov_inc3", "@hov2_inc1", "@hov2_inc2", "@hov2_inc3", "@hov3_inc1", "@hov3_inc2", "@hov3_inc3", "@av_sov_inc1", "@av_sov_inc2", "@av_sov_inc3", "@av_hov2_inc1", "@av_hov2_inc2", "@av_hov2_inc3", "@av_hov3_inc1", "@av_hov3_inc2", "@av_hov3_inc3", "@tnc_inc1", "@tnc_inc2", "@tnc_inc3", "@mveh", "@hveh", "@bveh"], "agency_lookup": {"1": "King County Metro", "2": "Pierce Transit", "3": "Community Transit", "4": "Kitsap Transit", "5": "Washington Ferries", "6": "Sound Transit", "7": "Everett Transit"}, "emissions_scenario": "standard", "tot_veh_model_base_year": 3185281, "speed_bins": [-999999.0, 2.5, 7.5, 12.5, 17.5, 22.5, 27.5, 32.5, 37.5, 42.5, 47.5, 52.5, 57.5, 62.5, 67.5, 72.5, 999999.0], "fac_type_lookup": {"0": 0, "1": 4, "2": 4, "3": 5, "4": 5, "5": 5, "6": 3, "7": 5, "8": 0}, "tod_lookup": {"5to9": 5, "9to15": 9, "15to18": 15, "18to20": 18, "20to5": 20}, "summer_list": [87], "special_route_lookup": {"1671": "A-Line Rapid Ride", "1672": "B-Line Rapid Ride", "1673": "C-Line Rapid Ride", "1674": "D-Line Rapid Ride", "1675": "E-Line Rapid Ride", "1677": "H-Line Rapid Ride", "4950": "Central Link", "6995": "Tacoma Link", "6998": "Sounder South", "6999": "Sounder North", "3701": "Swift Blue Line", "3702": "Swift Green Line"}}
input_config = {"debug_skims_and_paths": False, "model_year": "2023", "base_year": "2023", "landuse_inputs": "23_on_23_v3", "network_inputs": "base_year_2023_final", "db_name": "soundcast_inputs_2023.db", "soundcast_inputs_dir": "R:/e2projects_two/SoundCast/Inputs/rtp_2026_2050", "abm_model": "daysim", "run_accessibility_calcs": False, "run_setup_emme_project_folders": False, "run_setup_emme_bank_folders": False, "run_copy_scenario_inputs": False, "run_import_networks": False, "run_skims_and_paths_free_flow": False, "run_skims_and_paths": False, "run_truck_model": False, "run_supplemental_trips": False, "run_daysim": False, "run_summaries": True, "include_av": False, "include_tnc": True, "tnc_av": False, "include_tnc_to_transit": False, "include_knr_to_transit": False, "include_delivery": False, "include_telecommute": True, "run_integrated": False, "should_build_shadow_price": False, "delete_banks": False, "include_tnc_emissions": True, "add_distance_pricing": False, "distance_rate_dict": {"am": 13.5, "md": 8.5, "pm": 13.5, "ev": 8.5, "ni": 8.5}}


In [2]:
import pandas as pd
import polars as pl
from pathlib import Path
import util

pd.set_option('display.float_format', '{:,.0f}'.format)

In [3]:
def mic_equity_geog(df_in, equity_group):

    df_inc = df.pivot_table(
        index='person_work_mic',
        columns=equity_group,
        values='psexpfac',
        aggfunc='sum'
    ).reset_index()

    df_inc = df_inc.rename_axis(None, axis=1)
    pd.set_option('display.float_format', '{:,.0f}'.format)
    df_inc['Percent in Equity Geog'] = df_inc[1] / (df_inc[0] + df_inc[1])
    df_inc['Percent in Equity Geog'] = df_inc['Percent in Equity Geog'].fillna(0)
    df_inc['Percent in Equity Geog'] = (df_inc['Percent in Equity Geog'] * 100).round(1).astype(str) + '%'
    df_inc.sort_values(by=0, ascending=False)

    df_inc.rename(columns={'person_work_mic': 'MIC Work Location',
                        0: 'Below Regional Average', 
                                1: 'Above Regional Average', 
                                2: 'Higher Share of Equity Population'}, inplace=True)
    df_inc.index = df_inc['MIC Work Location']
    df_inc.drop('MIC Work Location', axis=1, inplace=True)
    # Percent of people of color
    
    return df_inc

## Workers in Manufacturing-Industrial Centers (MICs)

In [4]:
df = pd.read_csv(util.output_path / 'agg/dash/mic_workers.csv')

In [5]:
df = pd.read_csv(util.output_path / 'agg/dash/mic_workers.csv')
df = df[df['pwtyp'].isin(['Paid Full-Time Worker', 'Paid Part-Time Worker'])]
df.loc[df['person_work_mic'].isnull(), 'person_work_mic'] = 'Outside MIC'
_df = df[['psexpfac','person_work_mic']].groupby('person_work_mic').sum()[['psexpfac']]
_df.index
_df.rename(columns={'psexpfac': 'Total Workers'}, inplace=True)
_df


,Total Workers
person_work_mic,
Ballard-Interbay,"15,842"
Cascade,"8,697"
Duwamish,"57,364"
Frederickson,"4,055"
Kent MIC,"42,295"
North Tukwila,"6,289"
Outside MIC,"2,084,325"
Paine Field / Boeing Everett,"35,586"
Port of Tacoma,"9,495"


In [6]:
mic_equity_geog(df, "hh_efa_pov200").loc[_df.index]

,Below Regional Average,Above Regional Average,Higher Share of Equity Population,Percent in Equity Geog
person_work_mic,,,,
Ballard-Interbay,"11,510","2,882","1,450",20.0%
Cascade,"5,026","2,579","1,092",33.9%
Duwamish,"34,026","14,837","8,501",30.4%
Frederickson,"2,116","1,430",509,40.3%
Kent MIC,"21,380","12,357","8,558",36.6%
North Tukwila,"3,109","1,940","1,240",38.4%
Outside MIC,"1,299,100","521,777","263,448",28.7%
Paine Field / Boeing Everett,"20,693","9,636","5,257",31.8%
Port of Tacoma,"4,351","3,087","2,057",41.5%


In [7]:
mic_equity_geog(df, "hh_efa_poc").loc[_df.index]

,Below Regional Average,Above Regional Average,Higher Share of Equity Population,Percent in Equity Geog
person_work_mic,,,,
Ballard-Interbay,"10,030","3,800","2,012",27.5%
Cascade,"7,603",933,161,10.9%
Duwamish,"26,049","16,290","15,025",38.5%
Frederickson,"2,254","1,248",553,35.6%
Kent MIC,"14,943","12,833","14,519",46.2%
North Tukwila,"2,157","1,787","2,345",45.3%
Outside MIC,"1,124,348","610,385","349,592",35.2%
Paine Field / Boeing Everett,"20,131","12,343","3,112",38.0%
Port of Tacoma,"4,539","3,118","1,838",40.7%


In [8]:
mic_equity_geog(df, "hh_efa_lep").loc[_df.index]

,Below Regional Average,Above Regional Average,Higher Share of Equity Population,Percent in Equity Geog
person_work_mic,,,,
Ballard-Interbay,"10,912","2,831","2,099",20.6%
Cascade,"7,366",930,401,11.2%
Duwamish,"30,709","12,555","14,100",29.0%
Frederickson,"3,147",582,326,15.6%
Kent MIC,"17,630","10,508","14,157",37.3%
North Tukwila,"2,628","1,514","2,147",36.6%
Outside MIC,"1,298,439","440,632","345,254",25.3%
Paine Field / Boeing Everett,"19,153","8,370","8,063",30.4%
Port of Tacoma,"6,264","1,908","1,323",23.3%


## Commute Characteristics

In [9]:
df = pd.read_csv(util.output_path / 'agg/dash/tour_mic_dest.csv')
# Commute tours to MICs
df = df[df['pdpurp']=="Work"]
df.loc[df['tour_d_mic'].isnull(), 'tour_d_mic'] = 'Outside MIC'

# tmodetp by tour_d_mic
df_mode = pd.pivot_table(
    df,
    index='tour_d_mic',
    columns='tmodetp',
    values='toexpfac',
    aggfunc='sum'
)

# mode share by tour_d_mic
df_mode = df_mode.div(df_mode.sum(axis=1), axis=0)
df_mode = df_mode.fillna(0)
df_mode = df_mode.reset_index()
df_mode.rename(columns={'tour_d_mic': 'MIC Work Location'}, inplace=True)
df_mode.index = df_mode['MIC Work Location']
df_mode.drop('MIC Work Location', axis=1, inplace=True)

df_mode['Transit'] = df_mode['Transit']+df_mode['Park']
df_mode['HOV'] = df_mode['HOV2'] + df_mode['HOV3+']
df_mode['Walk/Bike/Other'] =  df_mode['Walk']+df_mode['Bike']+df_mode['TNC']
# df_mode = df_mode.sort_values(by='Drive Alone', ascending=False)
# Show results as percentages
df_mode = df_mode.map(lambda x: f"{x:.1%}")
# df_mode.sort_values(by='Walk', ascending=False, inplace=True)


df_mode[['SOV', 'HOV', 'Transit', 'Walk/Bike/Other']]

tmodetp,SOV,HOV,Transit,Walk/Bike/Other
MIC Work Location,,,,
Ballard-Interbay,65.3%,25.3%,1.7%,7.6%
Cascade,66.1%,30.7%,0.1%,3.1%
Duwamish,65.9%,28.3%,3.0%,2.8%
Frederickson,67.0%,31.7%,0.0%,1.2%
Kent MIC,66.0%,31.2%,1.3%,1.6%
North Tukwila,66.2%,30.1%,1.7%,2.1%
Outside MIC,61.7%,27.6%,3.5%,7.2%
Paine Field / Boeing Everett,67.7%,30.1%,0.3%,1.9%
Port of Tacoma,67.6%,30.5%,0.5%,1.3%


In [10]:
# Commute distance
pd.set_option('display.float_format', '{:,.1f}'.format)

df = pd.read_csv(util.output_path / 'agg/dash/tour_distance_mic.csv')
df = df[df['pdpurp'] == 'Work']
df['wt_dist'] = df['tautodist_bin'] * df['toexpfac']
df = df.groupby("person_work_mic").agg(
    {'wt_dist': 'sum', 'toexpfac': 'sum'}
)
df['average_distance'] = df['wt_dist']/df['toexpfac']
df[['average_distance']]

,average_distance
person_work_mic,
Ballard-Interbay,10.3
Cascade,10.7
Duwamish,12.8
Frederickson,8.8
Kent MIC,12.5
North Tukwila,12.7
Paine Field / Boeing Everett,11.2
Port of Tacoma,11.7
Puget Sound Industrial Center- Bremerton,12.5


## Demographics

In [11]:
# Demographics 
df = pd.read_csv(util.output_path / 'agg/dash/mic_workers.csv')
df['income_wt'] = df['hhincome_thousands'] * df['psexpfac']
df['income_wt'] = df['income_wt'].fillna(0)
df = df.groupby('person_work_mic').agg(
    {'psexpfac': 'sum', 'income_wt': 'sum'}
).reset_index()
df["avg_weighted_hh_income"] = df['income_wt']/df['psexpfac']
df['avg_weighted_hh_income'] = df['avg_weighted_hh_income'].apply(lambda x: f"${x:,.0f}")
df[['person_work_mic', 'avg_weighted_hh_income']]

,person_work_mic,avg_weighted_hh_income
0,Ballard-Interbay,"$189,432"
1,Cascade,"$153,292"
2,Duwamish,"$183,168"
3,Frederickson,"$143,398"
4,Kent MIC,"$164,779"
5,North Tukwila,"$164,792"
6,Paine Field / Boeing Everett,"$157,864"
7,Port of Tacoma,"$146,415"
8,Puget Sound Industrial Center- Bremerton,"$125,341"
9,Sumner Pacific,"$151,403"


In [12]:
df = pd.read_csv(util.output_path / 'agg/dash/mic_workers.csv')
df['hhsize_wt'] = df['hhsize'] * df['psexpfac']
df['hhsize_wt'] = df['hhsize_wt'].fillna(0)
df = df.groupby('person_work_mic').agg(
    {'psexpfac': 'sum', 'hhsize_wt': 'sum'}
)
df['avg_wt_hhsize'] = df['hhsize_wt']/df['psexpfac']
df = df.reset_index()
df[['person_work_mic', 'avg_wt_hhsize']]

,person_work_mic,avg_wt_hhsize
0,Ballard-Interbay,2.9
1,Cascade,3.5
2,Duwamish,3.1
3,Frederickson,3.6
4,Kent MIC,3.4
5,North Tukwila,3.2
6,Paine Field / Boeing Everett,3.2
7,Port of Tacoma,3.4
8,Puget Sound Industrial Center- Bremerton,3.0
9,Sumner Pacific,3.4


In [13]:
df = pd.read_csv(util.output_path / 'agg/dash/mic_workers.csv')
# Pivot table of worker type by person_work_mic
df_worker_type = df.pivot_table(
    index='person_work_mic',
    columns='pwtyp',
    values='psexpfac',
    aggfunc='sum'
)

# Share by worker type

df_worker_type = df_worker_type.div(df_worker_type.sum(axis=1), axis=0)
df_worker_type.rename(columns={'Paid Full-Time Worker': 'Full Time', 'Paid Part-Time Worker': 'Part Time'})
df_worker_type.map(lambda x: f"{x:.1%}")

pwtyp,Paid Full-Time Worker,Paid Part-Time Worker
person_work_mic,,
Ballard-Interbay,83.2%,16.8%
Cascade,74.1%,25.9%
Duwamish,87.1%,12.9%
Frederickson,69.0%,31.0%
Kent MIC,85.3%,14.7%
North Tukwila,81.1%,18.9%
Paine Field / Boeing Everett,83.4%,16.6%
Port of Tacoma,83.8%,16.2%
Puget Sound Industrial Center- Bremerton,64.2%,35.8%
